# Marker Repo - annotation

In this notebook, clustered h5ad files can be annotated using the marker repo.

## Loading packages

In [ ]:
import markerrepo.marker_repo as mr
import markerrepo.wrappers as wrap
import markerrepo.annotation as annot
import scanpy as sc

%load_ext autoreload
%autoreload 2

## Settings

Specify path of the cloned repository and the h5ad file which is going to be annotated.

In [ ]:
repo_path = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features"
h5ad_path = "/mnt/workspace/mkessle/master/refdata/hs.h5ad"

Load anndata and list all possible settings.

In [ ]:
adata = sc.read_h5ad(h5ad_path)
annot.list_possible_settings(repo_path, adata=adata)

Enter annotation settings.

In [ ]:
# Taxonomy ID or Organism Name
# e.g., "human" or 9606
organism = "human"

# Column in .obs table where ranked genes groups are stored
# e.g., "rank_genes_groups"
# Enter None if no ranking has been performed yet
rank_genes_column = None

# Column in .var table where gene symbols or Ensembl IDs are stored
# Enter None if the index column of the .var table already has gene symbols or Ensembl IDs
# that you want to use for your annotation
genes_column = None

# The .obs table column of the clustering you want to annotate
# e.g., "leiden" or "louvain"
column = "cell_types"

# Specify whether your index of the .var tables are Ensembl IDs (True) or gene symbols (False)
ensembl = mr.check_ensembl(adata)

# Name of the column to add with the final cell type annotation
# If None, all annotation columns will be kept
celltype_column_name = "pred_celltype"

# Whether to delete the created marker lists after annotation or not
delete_lists = True

**Specify Marker Lists for Annotation using column specific terms:**

- `key`: Specify the column to search in. Use `None` to search across all columns. 
  - Example columns include `Source`, `Organism name`, etc.
- `value`: Define your search terms. Use `-` to exclude keywords and `+` to ensure the keyword must be present.
  - Separate multiple keywords with a comma. For example: `["+panglao.se", "+mouse"]` to include lists from 'panglao.se' and related to 'mouse'.

In [ ]:
column_specific_terms={"Source":"panglao", "Organism name":"human"}

Adjust various settings for marker lists, like 'style' or 'file_name', otherwise the default settings will be used. A dictionary corresponds to a marker list.

Example:
```python
settings = [
    {
        "style": "two_column",
        "file_name": "basic_markers"
    },
    {
        "style": "score",
        "column_specific_terms": {
            "Source": "panglao",
            "Tissue": "heart"
        },
        "file_name": "heart_panglao"
    },
    {
        "force_homology": True,
        "file_name": "homology_markers"
    }
]


In [ ]:
settings = [{"style":"two_column", "file_name":"two_column"},
            {"style":"score", "file_name":"score"},
            {"style":"score", "column_specific_terms":{"Source":"panglao", "Tissue":"heart"}, 
             "file_name":"heart_panglao"},
            {"style":"score", "column_specific_terms":{"Tissue":"heart"}, "file_name":"heart_all"},
            {"force_homology":True, "file_name":"homol"}]

Validate input

In [ ]:
annot.validate_settings(settings=settings, repo_path=repo_path, adata=adata, organism=organism, 
                        rank_genes_column=rank_genes_column, genes_column=genes_column, column=column, ensembl=ensembl,
                        column_specific_terms=column_specific_terms)

## Prepare adata

### Set genes to index, if not already done.

In [ ]:
if genes_column:
    adata.var.reset_index(inplace=True)  # remove old index values and save them in the column ['index']
    adata.var.set_index(genes_column, inplace=True)  # set genes as index
    adata.var.index = adata.var.index.astype('str')  # to avoid index being categorical
    adata.var_names_make_unique(join='_')
    
    # update ensembl if gene identifier has changed
    ensembl = mr.check_ensembl(adata)
    
display(adata.var)

## Prepare annotation

### Ranking

Rank genes, if not already done.

In [ ]:
if not rank_genes_column:
    adata.uns['log1p']['base'] = None
    rank_genes_column = f'rank_genes_groups_{column}'
    print(f'Ranking genes groups for clusters using obs column {column}')
    sc.tl.rank_genes_groups(adata, groupby=f'{column}', use_raw=False, key_added=rank_genes_column)

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata, standard_scale='var', n_genes=10, key=rank_genes_column, show=True)

## Create suitable marker list(s)

The paths of the marker lists will be stored in the <b>marker_lists</b> variable. They will work as input for the actual cell type annotation of the next cell.

In [ ]:
marker_lists = wrap.create_multiple_marker_lists(settings=settings, repo_path=repo_path, organism=organism, 
                                                 ensembl=ensembl, column_specific_terms=column_specific_terms,
                                                 show_lists=True)

## Annotate adata using the created list(s)

In [ ]:
wrap.run_annotation(adata, SCSA=False, marker_lists=marker_lists, reference_obs=column, show_comparison=True,
                    clustering_column=column, rank_genes_column=rank_genes_column, ignore_overwrite=True,
                    verbose=False, show_plots=False, show_ct_tables=False, celltype_column_name=celltype_column_name)

In [ ]:
adata.obs

In [ ]:
import os

if delete_lists:
    for file_path in marker_lists:
        try:
            os.remove(file_path)
            print(f"File deleted: {file_path}")
        except FileNotFoundError:
            print(f"File not found: {file_path}")
        except Exception as e:
            print(f"Error deleting {file_path}: {e}")
